In [1]:
### REQUIRES conda_geo environment from Geo_requirements.yml

import geopandas as gpd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray
import xdem
import os

## Orography and calculation of terrain-related parameters

### Obtaining ERA5Land Orography from geopotential
The ERA5Land geopotential is a global land parameter which is time-invariant. It is intended as the geopotential value at the terrain surface with a 0.1°x0.1° resolution (see the documentation https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A+data+documentation#ERA5Land:datadocumentation-parameterlistingParameterlistings).
It can be downloaded from here https://confluence.ecmwf.int/download/attachments/140385202/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc?version=1&modificationDate=1591983422003&api=v2 .

From this field, we can obtain the orography of the terrain (elevation, so equivalent to a DEM) by dividing by the gravity acceleration. Here we use the averaged value of 9.81 m/s^2. Here we open it with rioxarray's open_rasterio method and not with xarray in order to assign to the output a geographic projection.

In [2]:
# Load the geopotential and compute the orography
era5l_geopotential = rioxarray.open_rasterio('./DEMdata/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc')
era5l_orography = era5l_geopotential/9.81

# Use rioxarray to assign a geographic projection to the obtained orography map, and save it using rioxarray methods.
era5l_orography = era5l_orography.rio.write_crs('EPSG:4326') # WGS84, the World Geodetic System
era5l_orography.rio.to_raster('./DEMdata/era5land_orography_global_wgs84.tif')

In [3]:
country = 'croatia' #e.g. cuba, croatia
shapefile_path = './world_map/ne_50m_admin_0_countries.shp'
geometries = gpd.read_file(shapefile_path)
geometries = geometries[geometries['NAME_EN'] == f'{country.capitalize()}']['geometry']

### Open orography with XDEM and calculate the aspect

We calculate the aspect (direction faced by each slope, in degrees from North) using the XDEM package.

In [4]:
# Load the orography using XDEM and use the .aspect() method.
era5l_orography = xdem.DEM('./DEMdata/era5land_orography_global_wgs84.tif')
aspect = era5l_orography.aspect() # Importantly, XDEM preserves the geographic projection assigned to the input.

# Save the output with the .save() method of XDEM.
aspect.save('./DEMdata/era5land_aspect_global_wgs84.tif')

### Cropping of ERA5 geodata

In [5]:
output_path = f'./DEMdata/{country}/'
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

### Crop to country
Load country geometry to use it to crop the DEM and aspect data.
Country geometries are derived from Natural Earth, and in particular from this dataset
https://github.com/nvkelso/natural-earth-vector/blob/master/50m_cultural/ne_50m_admin_0_countries.shp

In [6]:
## Some functions to handle cropping

def _roll_xdf(xdf):
    """
    Rolls the longitude values of the xarray DataArray from 0,360 to -180,180.

    Parameters:
        xdf (xarray.DataArray): The input xarray DataArray.

    Returns:
        xarray.DataArray: The rolled xarray DataArray with corrected longitude values.
    """
    xdf.coords['longitude'] = (xdf.coords['longitude'] + 180) % 360 - 180
    xdf = xdf.sortby(xdf.longitude)

    return xdf

def _rename_coords(xdf):
    """
    Renames the coordinate variables in the given xarray dataset or data array.

    Parameters:
    xdf (xarray.Dataset or xarray.DataArray): The input xarray dataset or data array.

    Returns:
    xarray.Dataset or xarray.DataArray: The input xarray object with renamed coordinate variables.

    """
    if 'x' in xdf.coords or 'lon' in xdf.coords:
        if 'x' in xdf.coords:
            xdf = xdf.rename({'x':'longitude', 'y':'latitude'})
        else:
            xdf = xdf.rename({'lon':'longitude', 'lat':'latitude'})
    return xdf

def _drop_coords(xdf):
    """
    Drops the 'spatial_ref' coordinate from the xarray dataset and squeezes the dataset if 0d coordinates are present.

    Parameters:
    xdf (xarray.Dataset): The input xarray dataset.

    Returns:
    xarray.Dataset: The modified xarray dataset with the 'spatial_ref' coordinate dropped and squeezed.
    """
    if 'spatial_ref' in xdf.coords:
        xdf = xdf.drop('spatial_ref')

    xdf = xdf.squeeze(drop=True)
    return xdf

def fix_coords(xdf):
    """
    Fixes the coordinates in the given DataFrame.

    Parameters:
    xdf (DataFrame): The input DataFrame containing coordinates.

    Returns:
    DataFrame: The DataFrame with fixed coordinates.
    """
    xdf = _rename_coords(xdf)
    xdf = _drop_coords(xdf)


    xdf['latitude'] = xdf.latitude.astype('float32')
    xdf['longitude'] = xdf.longitude.astype('float32')

    if not any(xdf.longitude < 0):
        xdf = _roll_xdf(xdf)

    return xdf


def clip_country(xdf_to_cut, country_geometry):
    """
    Clips a given xarray Dataset to the shape of a specified country.

    Parameters:
    xdf_to_cut (xarray.DataArray or xarray.Dataset): The data to be clipped.
    country_geometry (GeoDataFrame or similar): The geometric boundaries of the country to clip to. It is the actual shape, not the bounding box.

    Returns:
    xarray.Dataset: The clipped data, constrained to the specified country geometry.
    """

    def _crs_check(xdf):
        """
        Ensures the input data has the correct spatial dimensions (in terms of naming) and coordinate reference system (CRS).

        Parameters:
        xdf (xarray.DataArray or xarray.Dataset): The data to check and adjust.

        Returns:
        xarray.Dataset: The data with spatial dimensions set and CRS written.
        """
        xdf = xdf.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
        xdf = xdf.rio.write_crs("epsg:4326", inplace=True)
        return xdf

    xdf_to_cut_fix = fix_coords(xdf_to_cut)
    xdf_to_cut_crs = _crs_check(xdf_to_cut_fix)

    return xdf_to_cut_crs.rio.clip(country_geometry)

In [7]:
### For cropping the aspect: Use this if loading from a tif
era5l_aspect    = rioxarray.open_rasterio('./DEMdata/era5land_aspect_global_wgs84.tif').to_dataset(name = 'aspect')
era5l_orography = rioxarray.open_rasterio('./DEMdata/era5land_orography_global_wgs84.tif').to_dataset(name = 'orography')

cropped_aspect    = clip_country(era5l_aspect, geometries)
cropped_orography = clip_country(era5l_orography, geometries)
cropped_orography = cropped_orography.rename({'orography':'z'})

cropped_aspect.to_netcdf(output_path+f'era5land_aspect_{country}.nc')
cropped_orography.to_netcdf(output_path+f'era5land_orography_{country}.nc')

## Open Topography - Copernicus GLO-90 Digital Elevation Model
High Resolution Topography Data and Tools: https://portal.opentopography.org/raster?opentopoID=OTSDEM.032021.4326.1

Select the region, the output format and the visualization type (slope, aspect, hillshade,...)
Here, we downloaded the orography in the DEMdata folder and now we calculate the aspect.

In [8]:
glo90_orography = rioxarray.open_rasterio(f'./DEMdata/{country}/glo90_orography_{country}.tif').to_dataset(name = 'z')

cropped_orography_glo = clip_country(glo90_orography, geometries)
cropped_orography_glo.to_netcdf(output_path+f'glo90_orography_{country}.nc')

### Computing aspect from GLO90 orography

In [9]:
orography = xdem.DEM(f'./DEMdata/{country}/glo90_orography_{country}.nc')
aspect = orography.aspect()
aspect.save(output_path+f'glo90_aspect_{country}.tif') # XDEM cannot save directly in netcdf format.

# As the aspect obtained couldn't be saved in NetCDF format, let's open it with Xarray and save it with the correct format.
# NetCDF is needed for creating the feature set for downscaling.
aspect = xr.open_dataset(f'./DEMdata/{country}/glo90_aspect_{country}.tif')
aspect = aspect.rename({'x':'longitude','y':'latitude','band_data':'aspect'})
aspect.to_netcdf(output_path+f'glo90_aspect_{country}.nc')